## tl;dr

GRU 与 CfC 的 B2 acquisition funnel 都主要失败在搜索几何：weak-recovery 人口中的 geometry miss 分别为 119/135（88.15%）和 144/162（88.89%）。现有结果没有观察到 sampling loss，但 CfC 只有 3 个 weak-recovery pool 行、GRU 没有 pool 行，因此只能判断‘当前主要矛盾不是采样’，不能证明固定采样已经普遍可靠。下一步应保持 B0、extension-only、128/128 配额和采样不变，先验证 bounded adaptive shell。

## Context & Methods

本 notebook 复算服务器 commit `e9a2d6d` 导出的两个 epoch60、mini-dev、candidate0 acquisition-only JSON。分析单位是唯一 tracklet/frame 行。weak recovery 定义为 `base_target_count <= 2`；strict miss 定义为 `base_target_count == 0`。

### Key Assumptions

- 两份 JSON 是用户从对应 GRU/CfC `last.ckpt` 原样复制的报告。
- 汇总 JSON 足以定位主漏失阶段，但不能替代逐帧 CSV 的配对分析。
- `sampling_point_recall` 的点数分子/分母不在 JSON 中，因此这里只复核行级分解和可由行数重建的 recall。

## Data

In [1]:
import json
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'models').exists():
    project_root = project_root.parent
assert (project_root / 'models').exists(), 'CT-SeqTrack project root not found'

source_paths = {
    'GRU': project_root / 'artifacts/ct_checks/b1_gru_acquisition_funnel_e9a2d6d.json',
    'CfC': project_root / 'artifacts/ct_checks/b1_cfc_acquisition_funnel_e9a2d6d.json',
}
reports = {name: json.loads(path.read_text(encoding='utf-8'))
           for name, path in source_paths.items()}
[(name, report['diagnostic_rows'], report['diagnostic_tracklets'],
  report['source_checkpoint_epoch']) for name, report in reports.items()]

[('GRU', 311, 14, 60), ('CfC', 310, 14, 60)]

## Results

In [2]:
def close_or_both_none(actual, expected, tolerance=1e-12):
    if actual is None or expected is None:
        return actual is None and expected is None
    return abs(actual - expected) <= tolerance

for backend, report in reports.items():
    assert report['population'] == 'dev_candidate0'
    assert report['source_checkpoint_epoch'] == 60
    assert report['diagnostic_tracklets'] == 14
    assert sum(report['acquisition_stage_counts'].values()) == report['diagnostic_rows']
    for population_key in ('acquisition_weak_recovery', 'acquisition_strict_miss'):
        population = report[population_key]
        rows = population['rows']
        support_rows = rows - population['geometry_miss_rows']
        pool_rows = support_rows - population['no_novel_target_rows']
        sampled_rows = population['retained_rows']
        assert (population['geometry_miss_rows']
                + population['no_novel_target_rows']
                + population['sampling_loss_rows']
                + population['retained_rows']) == rows
        expected = {
            'support_row_recall': support_rows / rows if rows else None,
            'pool_row_recall': pool_rows / support_rows if support_rows else None,
            'sampling_row_recall': sampled_rows / pool_rows if pool_rows else None,
            'end_to_end_row_retention': sampled_rows / rows if rows else None,
        }
        for metric, value in expected.items():
            assert close_or_both_none(population[metric], value), (backend, population_key, metric)
print('All row-level funnel reconciliations passed.')

All row-level funnel reconciliations passed.


In [3]:
summary_rows = []
for backend, report in reports.items():
    for population_label, population_key in (
            ('weak_recovery', 'acquisition_weak_recovery'),
            ('strict_miss', 'acquisition_strict_miss')):
        population = report[population_key]
        rows = population['rows']
        support_rows = rows - population['geometry_miss_rows']
        pool_rows = support_rows - population['no_novel_target_rows']
        summary_rows.append({
            'backend': backend,
            'population': population_label,
            'rows': rows,
            'geometry_miss_rows': population['geometry_miss_rows'],
            'geometry_miss_share': population['geometry_miss_rows'] / rows,
            'support_rows': support_rows,
            'support_row_recall': population['support_row_recall'],
            'no_novel_target_rows': population['no_novel_target_rows'],
            'pool_rows': pool_rows,
            'pool_row_recall': population['pool_row_recall'],
            'sampling_loss_rows': population['sampling_loss_rows'],
            'retained_rows': population['retained_rows'],
            'sampling_row_recall': population['sampling_row_recall'],
            'end_to_end_row_retention': population['end_to_end_row_retention'],
        })
summary = pd.DataFrame(summary_rows)
summary

,backend,population,rows,geometry_miss_rows,geometry_miss_share,support_rows,support_row_recall,no_novel_target_rows,pool_rows,pool_row_recall,sampling_loss_rows,retained_rows,sampling_row_recall,end_to_end_row_retention
0,GRU,weak_recovery,135,119,0.881481,16,0.118519,16,0,0.000000,0,0,NaN,0.000000
1,GRU,strict_miss,104,104,1.000000,0,0.000000,0,0,NaN,0,0,NaN,0.000000
2,CfC,weak_recovery,162,144,0.888889,18,0.111111,15,3,0.166667,0,3,1.0,0.018519
3,CfC,strict_miss,137,134,0.978102,3,0.021898,0,3,1.000000,0,3,1.0,0.021898


In [4]:
weak = summary[summary['population'] == 'weak_recovery'].copy()
weak[['backend', 'rows', 'geometry_miss_rows', 'geometry_miss_share',
      'support_rows', 'no_novel_target_rows', 'pool_rows',
      'sampling_loss_rows', 'retained_rows',
      'end_to_end_row_retention']]

,backend,rows,geometry_miss_rows,geometry_miss_share,support_rows,no_novel_target_rows,pool_rows,sampling_loss_rows,retained_rows,end_to_end_row_retention
0,GRU,135,119,0.881481,16,16,0,0,0,0.000000
2,CfC,162,144,0.888889,18,15,3,0,3,0.018519


## Takeaways

1. **搜索几何是当前主瓶颈。** weak recovery 中 GRU 有 88.15%、CfC 有 88.89% 的行在 raw endpoint+tube support 阶段仍完全没有目标点。strict miss 更明确：GRU 104/104 geometry miss；CfC 134/137 geometry miss。
2. **现有数据不支持优先修改采样。** 两组 `sampling_loss_rows` 都为 0；CfC 进入 weak-recovery pool 的 3 行全部被采样保留，GRU 则没有任何 weak-recovery pool 行。采样机会过少，所以结论是‘没有观察到采样丢失’，不是‘固定采样已经被充分证明可靠’。
3. **`no_novel_target` 是次级现象，且在 strict miss 中为 0。** GRU 的 16 行、CfC 的 15 行只出现在 base 已有 1–2 个目标点的 weak population；它们可能只是扩展区域重复覆盖已有 base return，随后被 extension-only 排除，并不能单独证明 pool 构造有 bug。
4. **下一步主方向应是 stable B0 base + bounded adaptive shell。** 第一轮只改变搜索几何，保持 extension-only、128/128 配额、固定采样和 soft-mean/voting 不变。几何改善后重新跑同一 funnel；只有 pool 机会显著增加后出现 sampling loss，才进入 relation-aware hybrid sampling。
5. **不能用这份 funnel 单独决定 GRU/CfC 胜负。** 两份汇总相差 1 行，缺少逐帧配对 CSV；CfC strict miss 多保留 3 行是小样本信号。后端仍应由 B1 的 paired RMSE/NLL/coverage 决定，B2 开发中不要同时改 backend 和 shell。